# Question 2

In [51]:
import pyomo.environ as pyo
import pandas as pd
# Use the HiGHS solver through Pyomo's APPSI interface
SOLVER = pyo.SolverFactory('appsi_highs')
assert SOLVER.available(), "HiGHS (appsi_highs) solver is not available."
print("Solver ready:", SOLVER)


Solver ready: <pyomo.contrib.appsi.base.SolverFactoryClass.register.<locals>.decorator.<locals>.LegacySolver object at 0x0000019C095F2490>


In [52]:
df = pd.read_csv("Gerrymandering.csv")
df = df.set_index("County Num")
df.head()

,County,D_j - R_j
County Num,,
1,Bernalillo,42941
2,Catron,-917
3,Chaves,-6650
4,Cibola,1941
5,Colfax,116


In [53]:
model = pyo.ConcreteModel()

model.I = pyo.RangeSet(1,3)
model.J = pyo.RangeSet(1,33)

# Decision variables: x[i,j] = if county j is assigned to district i 
model.x = pyo.Var(model.I, model.J, domain=pyo.Binary)
model.D = pyo.Var(model.I, domain=pyo.Reals)


model.county_difference = pyo.Param(model.J, initialize=df["D_j - R_j"].to_dict())

# Each county must be assigned to exactly one district
def unique_assignment(model, j):
    return sum(model.x[i,j] for i in model.I) == 1 

model.unique_assignment = pyo.Constraint(model.J, rule=unique_assignment)

# At least one county must be assigned to each district
def valid_district(model, i):
    return sum(model.x[i,j] for j in model.J) >= 1

model.valid_district = pyo.Constraint(model.I, rule=valid_district)

def district_difference(model, i):
    return sum(model.county_difference[j] * model.x[i,j] for j in model.J) == model.D[i]

model.district_difference = pyo.Constraint(model.I, rule=district_difference)

def winning_buffer(model, i):
    return model.D[i] >= 100

model.winning_buffer = pyo.Constraint(model.I, rule=winning_buffer)


In [54]:
model.obj = pyo.Objective(expr = model.D[2], sense = pyo.maximize)
results = SOLVER.solve(model)


In [55]:
print("Optimal Value", pyo.value(model.obj))
print("District Differences:")
for i in model.I:
    print(f"District {i}: {pyo.value(model.D[i])}")
    
for i in model.I:
    district_counties = []
    for j in model.J:
        if pyo.value(model.x[i,j]) == 1:  
            district_counties.append(j)
    print(f"District {i}: Counties = {df.loc[district_counties]}, Num = {len(district_counties)}")

Optimal Value 76192.0
District Differences:
District 1: 105.0
District 2: 76192.0
District 3: 100.0
District 1: Counties =                County  D_j - R_j
County Num                      
3              Chaves      -6650
11          Guadalupe        870
13            Hidalgo         99
18           McKinley       9943
19               Mora       1361
21               Quay       -812
25           San Juan     -13091
30               Taos       9145
32              Union       -760, Num = 9
District 2: Counties =                 County  D_j - R_j
County Num                       
1           Bernalillo      42941
6                Curry      -5194
7               DeBaca       -299
8             Dona Ana       9790
9                 Eddy      -6436
10               Grant       1723
12             Harding        -66
17                Luna        -81
20               Otero      -5504
23           Roosevelt      -2313
24            Sandoval       2707
26          San Miguel       6473
27    

\newpage

# Question 3

In [106]:
model = pyo.ConcreteModel()
model.I = pyo.RangeSet(1,9)
model.J = pyo.RangeSet(1,9)
model.K = pyo.RangeSet(1,9)

model.X = pyo.RangeSet(1,3)
model.Y = pyo.RangeSet(1,3)



model.x = pyo.Var(model.I, model.J, model.K, domain=pyo.Binary, initialize=0)


def unique_in_row(model,j,k):
    return sum(model.x[i,j,k] for i in model.I) == 1
model.unique_in_row = pyo.Constraint(model.J, model.K, rule=unique_in_row)

def unique_in_col(model,i,k):
    return sum(model.x[i,j,k] for j in model.J) == 1
model.unique_in_col = pyo.Constraint(model.I, model.K, rule=unique_in_col)

def unique_in_box(model,x,y, k):
    return sum(model.x[i,j,k] for i in range(3*x-2, 3*x+1) for j in range(3*y-2, 3*y+1)) == 1
model.unique_in_box = pyo.Constraint(model.X, model.Y, model.K, rule=unique_in_box)

def one_in_cell(model,i,j):
    return sum(model.x[i,j,k] for k in model.K) == 1
model.one_in_cell = pyo.Constraint(model.I, model.J, rule=one_in_cell)

inital_values = {
    (1,6,1) : 1, 
    (2,1,2) : 1, (2,2,7) : 1, (2,5,9) : 1, (2,7,5) : 1,
    (3,2,8) : 1, (3,6,5) : 1, (3,9,3) : 1,
    (4,3,8) : 1, (4,6,3) : 1, (4,8,2) : 1,
    (5,2,5) : 1, (5,4,1) : 1, (5,6,2) : 1, (5,8,9) : 1,
    (6,2,1) : 1, (6,5,5) : 1, (6,7,7) : 1,
    (7,1,5) : 1, (7,4,6) : 1, (7,8,3) : 1,
    (8,3,9) : 1, (8,5,1) : 1, (8,8,6) : 1, (8,9,2) : 1,
    (9,4,2) : 1
}

model.inital_values = pyo.Param(model.I, model.J, model.K, initialize=inital_values)
def set_initial_values(model, i, j, k):
    if (i,j,k) in model.inital_values:
        return model.x[i,j,k] == model.inital_values[i,j,k]
    return pyo.Constraint.Skip
model.set_intial_values = pyo.Constraint(model.I, model.J, model.K, rule=set_initial_values)

model.obj = pyo.Objective(expr = 0, sense=pyo.minimize)
results = SOLVER.solve(model)  


In [109]:
for i in model.I:
    row = []
    for j in model.J:
        value = 0
        for k in model.K:
            if pyo.value(model.x[i,j,k]) == 1:
                value = k
        row.append(value)
    print(f"Row {i} = {row}")

Row 1 = [6, 4, 5, 3, 7, 1, 2, 8, 9]
Row 2 = [2, 7, 3, 8, 9, 6, 5, 1, 4]
Row 3 = [9, 8, 1, 4, 2, 5, 6, 7, 3]
Row 4 = [4, 9, 8, 7, 6, 3, 1, 2, 5]
Row 5 = [7, 5, 6, 1, 4, 2, 3, 9, 8]
Row 6 = [3, 1, 2, 9, 5, 8, 7, 4, 6]
Row 7 = [5, 2, 7, 6, 8, 4, 9, 3, 1]
Row 8 = [8, 3, 9, 5, 1, 7, 4, 6, 2]
Row 9 = [1, 6, 4, 2, 3, 9, 8, 5, 7]


In [116]:
import numpy as np
arr = np.zeros((9,9), dtype=int)

for i in model.I:
    for j in model.J:
        for k in model.K:
            if pyo.value(model.x[i,j,k]) == 1:
                arr[i-1,j-1] = k

for i in range(9):
    print(len(set(arr[i])) == 9)  # Check rows
    print(len(set(arr[:,i])) == 9)  # Check columns
    print(len(set(arr[3*(i//3):3*(i//3)+3, 3*(i%3):3*(i%3)+3].flatten())) == 9)  # Check boxes

True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
